# Paper Model Comparison

This report compares LR/RF classifiers across all 32/64 train/evaluation window pairs, feature sets, and training regimes, then reports mean-window and whole_text document decisions. Scorers are Gemma 3 270M IT, Gemma 3 1B IT, LittleLamb, and SmolLM2 360M where cached artifacts exist. Only cached features/models are scored; there is no LLM inference or first-stage model training.

## Metric scope

Mixed and Mixed+BBC contain HG and MG examples and receive binary macro-F1. BBC-only is HG-only, so its report is false-positive rate (FPR), not macro-F1. Name That LLM is MG-only, so it reports MG detection rate only and uses scorer-tokenizer length ≥250. SemEval window labels are derived from gold token boundaries; windows crossing a boundary and documents whose annotation cannot be aligned are excluded and counted. SemEval and Name That LLM use the available train partitions for cross-dataset analysis, not official test-set estimates.

In [ ]:
import json
import os
from pathlib import Path
from itertools import product
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
from notebooks.utils.paper_model_comparison_utils import (
    binary_window_metrics, build_cohort_manifest, build_configuration_coverage,
    filter_window_rows_to_common_documents,
    label_semeval_windows, load_document_word_counts, load_evaluation_window_rows, load_processed_token_counts,
    normalize_t08_aggregation_labels,
    load_semeval_token_labels, score_prepared_window_partition, single_class_rates,
)
from notebooks.utils.document_window_evaluation_utils import (
    aggregate_window_predictions, evaluate_document_predictions, filter_common_documents,
)
from notebooks.utils.document_level_model_experiment_utils import (
    build_document_feature_table, fit_and_evaluate_document_models,
    load_validation_test_features, score_document_models, summarize_mg_detection,
)
from notebooks.utils.semeval_text_comparison_utils import plot_window_level_scores

DATA_ROOT = Path(os.environ.get('TEXT_DETECTION_DATA', 'D:/Projects/master_diploma/text-detection-data'))
RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
TRAINING_REGIME = 'Kaggle+MAGE+Outfox'
SCORERS = ('gemma_3_270m_it', 'gemma_3_1b_it', 'littlelamb', 'smollm2_360m')
MODEL_FAMILIES = ('logistic_regression', 'random_forest')
TRAIN_WINDOWS = (32, 64)
EVALUATION_WINDOWS = (32, 64)
WINDOWS = (32, 64)
FEATURE_SETS = ('scalar_compact', 'scalar_all', 'simple_similarity', 'extended')
TRAINING_REGIMES = ('Kaggle', 'MAGE', 'Outfox', 'Kaggle+MAGE+Outfox', 'Kaggle+MAGE+Outfox+BBC')
EVALUATION_REGIMES = ('Kaggle+MAGE+Outfox', 'Kaggle+MAGE+Outfox+BBC')
WHOLE_TEXT_TRAINING_REPRESENTATIONS = (32, 64, 'whole_text')
MIN_TOKEN_COUNT = 250
N2LLM_LIMIT = 1800
PREDICTION_ROOT = DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / RUN_ID / 'predictions'
REPORT_RUN_ID = os.environ.get('CHAPTER3_REPORT_RUN_ID', 'chapter3-restructured-2026-10-27')
OUTPUT_ROOT = DATA_ROOT / 'experiment-results' / 'report' / 'chapter3-restructured' / REPORT_RUN_ID
TABLE_ROOT = OUTPUT_ROOT / 'tables'
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
APPENDIX_ROOT = OUTPUT_ROOT / 'appendix'
MANIFEST_ROOT = OUTPUT_ROOT / 'manifests'
for directory in (TABLE_ROOT, FIGURE_ROOT, APPENDIX_ROOT, MANIFEST_ROOT): directory.mkdir(parents=True, exist_ok=True)

In [ ]:
feature_columns = {
    'scalar_compact': ['d_score_top20', 'perplexity', 'prompt_mu_score'],
    'scalar_all': ['d_score_top20', 'perplexity', 'prompt_mu_score', 'prompt_sigma_score', 'd_score_top20_coverage'],
    'simple_similarity': ['mean_candidate_similarity__mean', 'mean_candidate_similarity__std', 'top20_coverage__mean', 'top20_coverage__std', 'observed_logprob__mean', 'observed_logprob__std', 'mean_candidate_logprob_delta__mean', 'mean_candidate_logprob_delta__std'],
}
feature_columns['extended'] = feature_columns['scalar_all'] + feature_columns['simple_similarity']
table_t01_feature_sets = pd.DataFrame([
    {'feature_set': name, 'feature_count': len(columns), 'feature_columns': '|'.join(columns)}
    for name, columns in feature_columns.items()
])
table_t02_evaluation_units = pd.DataFrame([
    {'evaluation_mode': 'window', 'fit_unit': 'window row', 'prediction_unit': 'window', 'aggregation': 'none', 'metric_unit': 'window'},
    {'evaluation_mode': 'document_mean_window', 'fit_unit': 'window row', 'prediction_unit': 'document', 'aggregation': 'mean continuous window MG-score; threshold 0.5', 'metric_unit': 'document'},
    {'evaluation_mode': 'whole_text_window_trained', 'fit_unit': 'window row', 'prediction_unit': 'document', 'aggregation': 'one whole_text feature vector; no window-score aggregation', 'metric_unit': 'document'},
    {'evaluation_mode': 'whole_text_trained', 'fit_unit': 'whole_text vector', 'prediction_unit': 'document', 'aggregation': 'one whole_text feature vector; no window-score aggregation', 'metric_unit': 'document'},
])
display(table_t01_feature_sets)
display(table_t02_evaluation_units)

## Window-level metrics: Mixed and Mixed+BBC

Use saved test window predictions. Within each scorer and regime, all 8 LR/RF × train/evaluation-window configurations are restricted to common document IDs before macro-F1 is computed.

In [ ]:
main_window_frames = []
for regime, scorer, family, train_window, eval_window in tqdm(product(EVALUATION_REGIMES, SCORERS, MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS), desc='Loading main window predictions'):
    rows = load_evaluation_window_rows(
        PREDICTION_ROOT, run_id=RUN_ID, scorer_llm=scorer, model_family=family,
        training_window=train_window, evaluation_regime=regime, evaluation_window=eval_window, partition='test',
    )
    rows['window_length'] = eval_window
    main_window_frames.append(rows)
main_window_rows_available = pd.concat(main_window_frames, ignore_index=True)
main_window_rows = main_window_rows_available.copy()
main_window_rows, main_window_coverage = filter_window_rows_to_common_documents(
    main_window_rows, comparison_group_columns=('scorer_llm', 'evaluation_regime'),
    configuration_columns=('model_family', 'training_window', 'evaluation_window'),
)
window_level_main_metrics = binary_window_metrics(
    main_window_rows,
    group_columns=('evaluation_regime', 'scorer_llm', 'model_family', 'training_window', 'evaluation_window'),
)
display(window_level_main_metrics.sort_values(['evaluation_regime', 'scorer_llm', 'evaluation_window', 'model_family']))
display(main_window_coverage)

## Window-level feature and training-regime blocks

F compares all four feature sets at matched windows on Mixed test. S compares five training compositions on the same Mixed evaluation regime using Gemma 270M. The tables use common documents within each scorer/model/window comparison.

In [ ]:
feature_window_frames = []
for scorer, family, feature_set, window in tqdm(product(SCORERS, MODEL_FAMILIES, FEATURE_SETS, WINDOWS), desc='Comparing feature sets'):
    rows = load_evaluation_window_rows(
        PREDICTION_ROOT, run_id=RUN_ID, scorer_llm=scorer, model_family=family, feature_set=feature_set,
        training_regime=TRAINING_REGIME, training_window=window,
        evaluation_regime='Kaggle+MAGE+Outfox', evaluation_window=window, partition='test',
    )
    rows['feature_set'] = feature_set
    rows['window_length'] = window
    feature_window_frames.append(rows)
feature_window_rows_available = pd.concat(feature_window_frames, ignore_index=True)
feature_window_rows = feature_window_rows_available.copy()
feature_window_rows, feature_window_coverage = filter_window_rows_to_common_documents(
    feature_window_rows,
    comparison_group_columns=('scorer_llm', 'evaluation_regime', 'training_window', 'evaluation_window'),
    configuration_columns=('model_family', 'feature_set'),
)
feature_window_metrics = binary_window_metrics(
    feature_window_rows,
    group_columns=('scorer_llm', 'model_family', 'feature_set', 'training_window', 'evaluation_window'),
)
display(feature_window_metrics.sort_values(['scorer_llm', 'evaluation_window', 'model_family', 'feature_set']))
display(feature_window_coverage)

In [ ]:
training_regime_frames = []
for training_regime, family, window in tqdm(product(TRAINING_REGIMES, MODEL_FAMILIES, WINDOWS), desc='Comparing training compositions'):
    rows = load_evaluation_window_rows(
        PREDICTION_ROOT, run_id=RUN_ID, scorer_llm='gemma_3_270m_it', model_family=family,
        feature_set='simple_similarity', training_regime=training_regime, training_window=window,
        evaluation_regime='Kaggle+MAGE+Outfox', evaluation_window=window, partition='test',
    )
    rows['window_length'] = window
    training_regime_frames.append(rows)
training_regime_rows_available = pd.concat(training_regime_frames, ignore_index=True)
training_regime_rows = training_regime_rows_available.copy()
training_regime_rows, training_regime_coverage = filter_window_rows_to_common_documents(
    training_regime_rows,
    comparison_group_columns=('scorer_llm', 'evaluation_regime', 'model_family', 'evaluation_window'),
    configuration_columns=('training_regime',),
)
training_regime_metrics = binary_window_metrics(
    training_regime_rows,
    group_columns=('scorer_llm', 'model_family', 'training_regime', 'training_window', 'evaluation_window'),
)
display(training_regime_metrics.sort_values(['training_regime', 'model_family', 'evaluation_window']))
display(training_regime_coverage)

## Window-level single-class and SemEval metrics

BBC-only is HG-only, so report FPR. SemEval windows are assigned HG/MG only when fully on one side of the token boundary; overlapping boundary windows and unalignable annotation documents are listed in the exclusion table. Name That LLM is MG-only and uses the 250-token lower bound.

In [ ]:
bbc_window_frames = []
for scorer, family, train_window, eval_window in tqdm(product(SCORERS, MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS), desc='Scoring BBC windows'):
    scored = score_prepared_window_partition(
        DATA_ROOT, run_id=RUN_ID, scorer_llm=scorer, dataset='bbc_news_content', split='HG',
        model_family=family, training_regime=TRAINING_REGIME, training_window=train_window, evaluation_window=eval_window,
    )
    scored['comparison_dataset'] = 'BBC-only'
    scored['window_length'] = eval_window
    scored['report_scope'] = 'primary' if scorer in ('gemma_3_270m_it', 'littlelamb') else 'appendix_available'
    bbc_window_frames.append(scored)
bbc_window_rows_available = pd.concat(bbc_window_frames, ignore_index=True)
bbc_windows = bbc_window_rows_available.copy()
bbc_windows, bbc_coverage = filter_window_rows_to_common_documents(
    bbc_windows, comparison_group_columns=('scorer_llm', 'comparison_dataset'),
    configuration_columns=('model_family', 'training_window', 'evaluation_window'),
)
bbc_window_rates = single_class_rates(
    bbc_windows, group_columns=('scorer_llm', 'model_family', 'training_window', 'evaluation_window'), unit_name='window',
)
bbc_whole_text_frames = []
for scorer, family, train_representation in tqdm(product(SCORERS, MODEL_FAMILIES, WHOLE_TEXT_TRAINING_REPRESENTATIONS), desc='Scoring BBC whole-text documents'):
    rows = score_prepared_window_partition(
        DATA_ROOT, run_id=RUN_ID, scorer_llm=scorer, dataset='bbc_news_content', split='HG',
        model_family=family, training_regime=TRAINING_REGIME, training_window=train_representation, evaluation_window='whole_text',
    )
    rows['report_scope'] = 'primary' if scorer in ('gemma_3_270m_it', 'littlelamb') else 'appendix_available'
    bbc_whole_text_frames.append(rows)
bbc_whole_text_predictions = pd.concat(bbc_whole_text_frames, ignore_index=True)
bbc_whole_text_rates = single_class_rates(
    bbc_whole_text_predictions,
    group_columns=('scorer_llm', 'model_family', 'training_window', 'evaluation_window'), unit_name='document',
)

semeval_annotations = pd.read_csv(
    DATA_ROOT / 'raw' / 'semevals_2024-task8' / 'SemEval2024-task8' / 'subtaskC' / 'SemEval2024_task8_C_train_MG.csv'
)
semeval_window_frames = []
semeval_scored_frames = []
semeval_exclusion_frames = []
for scorer, family, window in tqdm(product(SCORERS, MODEL_FAMILIES, WINDOWS), desc='Scoring SemEval windows'):
    scored = score_prepared_window_partition(
        DATA_ROOT, run_id=RUN_ID, scorer_llm=scorer, dataset='semeval2024_task8_subtask_c_train', split='MG',
        model_family=family, training_regime=TRAINING_REGIME, training_window=window, evaluation_window=window,
    )
    semeval_scored_frames.append(scored)
    token_labels = load_semeval_token_labels(DATA_ROOT, scorer_llm=scorer)
    labeled, excluded = label_semeval_windows(scored, semeval_annotations, token_labels)
    for frame in (labeled, excluded):
        frame['scorer_llm'] = scorer
        frame['model_family'] = family
        frame['training_window'] = window
        frame['evaluation_window'] = window
    semeval_window_frames.append(labeled)
    semeval_exclusion_frames.append(excluded)
semeval_window_rows_available = pd.concat(semeval_window_frames, ignore_index=True)
semeval_windows = semeval_window_rows_available.copy()
semeval_windows, semeval_coverage = filter_window_rows_to_common_documents(
    semeval_windows, comparison_group_columns=('scorer_llm',),
    configuration_columns=('model_family', 'training_window', 'evaluation_window'),
)
semeval_window_metrics = binary_window_metrics(
    semeval_windows,
    group_columns=('scorer_llm', 'model_family', 'training_window', 'evaluation_window'), label_column='gold_label',
)
semeval_excluded_windows = pd.concat(semeval_exclusion_frames, ignore_index=True)
semeval_exclusion_summary = semeval_excluded_windows.groupby(
    ['scorer_llm', 'model_family', 'training_window', 'evaluation_window'], as_index=False
).agg(
    semeval_document_count=('text_id', 'nunique'),
    windows_seen=('window_count', 'sum'),
    boundary_or_unalignable_windows=('excluded_boundary_window_count', 'sum'),
    unalignable_document_count=('exclusion_reason', lambda values: int(values.ne('').sum())),
)

n2llm_metadata = pd.read_csv(
    DATA_ROOT / 'raw' / 'name-that-llm' / 'name_that_llm_train_MG.csv', dtype={'id': str, 'source_id': str},
)
n2llm_metadata['text_id'] = n2llm_metadata['id'].astype(str).str.strip()
N2LLM_SCORERS = ('gemma_3_270m_it', 'littlelamb')
n2llm_token_counts_by_scorer = {
    scorer: load_processed_token_counts(
        DATA_ROOT, scorer_llm=scorer, dataset='name_that_llm_train', split='MG',
    )
    for scorer in N2LLM_SCORERS
}
n2llm_window_frames = []
for scorer, family, train_window, eval_window in tqdm(product(N2LLM_SCORERS, MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS), desc='Scoring Name That LLM windows'):
    scored = score_prepared_window_partition(
        DATA_ROOT, run_id=RUN_ID, scorer_llm=scorer, dataset='name_that_llm_train', split='MG',
        model_family=family, training_regime=TRAINING_REGIME, training_window=train_window, evaluation_window=eval_window,
    )
    scored['text_id'] = scored['text_id'].astype(str)
    scored = scored.merge(n2llm_metadata[['text_id', 'source_llm', 'id']], on='text_id', how='left', validate='many_to_one')
    scored['id'] = scored['id'].astype(str).str.strip()
    scored = scored.merge(n2llm_token_counts_by_scorer[scorer], on='id', how='left', validate='many_to_one')
    if scored['token_count'].isna().any():
        raise ValueError('Name That LLM window predictions are missing scorer token counts')
    n2llm_window_frames.append(scored)
n2llm_window_rows_available = pd.concat(n2llm_window_frames, ignore_index=True)
n2llm_windows = n2llm_window_rows_available.copy()
n2llm_windows['comparison_dataset'] = 'Name That LLM (>=250 tokens)'
n2llm_windows, n2llm_window_coverage = filter_window_rows_to_common_documents(
    n2llm_windows, comparison_group_columns=('scorer_llm', 'comparison_dataset'),
    configuration_columns=('model_family', 'training_window', 'evaluation_window'),
)
n2llm_common_cohort_window_rows = n2llm_windows.copy()
n2llm_available_ids = sorted(n2llm_windows['text_id'].astype(str).unique(), key=int)
if len(n2llm_available_ids) < N2LLM_LIMIT:
    raise ValueError(f'expected {N2LLM_LIMIT} common Name That LLM documents, found {len(n2llm_available_ids)}')
n2llm_selected_ids = set(n2llm_available_ids[-N2LLM_LIMIT:])
n2llm_cohort_window_rows = n2llm_windows.loc[n2llm_windows['text_id'].astype(str).isin(n2llm_selected_ids)].copy()
n2llm_token_filter_coverage = (
    n2llm_cohort_window_rows.drop_duplicates(['model_family', 'training_window', 'evaluation_window', 'text_id'])
    .groupby(['model_family', 'training_window', 'evaluation_window'], as_index=False)
    .agg(cohort_document_count=('text_id', 'nunique'), eligible_long_document_count=('token_count', lambda values: int(values.ge(MIN_TOKEN_COUNT).sum())))
)
n2llm_windows = n2llm_cohort_window_rows.loc[n2llm_cohort_window_rows['token_count'].ge(MIN_TOKEN_COUNT)].copy()
n2llm_window_rates = single_class_rates(
    n2llm_windows, group_columns=('scorer_llm', 'model_family', 'training_window', 'evaluation_window'), unit_name='window',
)

single_class_window_rates = pd.concat([bbc_window_rates.assign(dataset='BBC-only'), n2llm_window_rates.assign(dataset='Name That LLM (>=250 tokens)')], ignore_index=True)
display(bbc_window_rates)
display(semeval_window_metrics)
display(semeval_excluded_windows.groupby(['scorer_llm', 'model_family', 'evaluation_window'], as_index=False).agg(
    semeval_documents=('text_id', 'nunique'), crossing_or_unalignable_windows=('excluded_boundary_window_count', 'sum'),
))
display(n2llm_window_rates)

## Document-level Mixed and Mixed+BBC results

Document LR and EBM are fitted separately for each scorer, first-stage classifier family, and matched window on first-stage validation profiles. Metrics are computed once on common held-out test documents. The baseline thresholds the mean window score at 0.5.

In [ ]:
validation_by_config = {}
test_by_config = {}
config_dimensions = {}
for regime, scorer, family, train_window, eval_window in tqdm(product(EVALUATION_REGIMES, SCORERS, MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS), desc='Loading document validation/test profiles'):
    key = f'{regime}__{scorer}__{family}__{train_window}__{eval_window}'
    validation_docs, test_docs = load_validation_test_features(
        PREDICTION_ROOT, run_id=RUN_ID, scorer_llm=scorer, model_family=family,
        feature_set='simple_similarity', training_regime=TRAINING_REGIME, training_window=train_window,
        evaluation_regime=regime, evaluation_window=eval_window, document_threshold=0.5,
    )
    validation_by_config[key] = validation_docs
    test_by_config[key] = test_docs
    config_dimensions[key] = {'evaluation_regime': regime, 'scorer_llm': scorer, 'model_family': family, 'training_window': train_window, 'evaluation_window': eval_window}
validation_common = filter_common_documents(validation_by_config, common_group_columns=('evaluation_regime', 'scorer_llm'))
test_common = filter_common_documents(test_by_config, common_group_columns=('evaluation_regime', 'scorer_llm'))
document_metric_frames = []
document_test_prediction_frames = []
mixed_models_by_base_config = {}
for key in validation_common:
    predictions, metrics, fitted = fit_and_evaluate_document_models(validation_common[key], test_common[key], random_state=42)
    dimensions = config_dimensions[key]
    document_metric_frames.append(metrics.assign(**dimensions))
    document_test_prediction_frames.append(predictions.assign(**dimensions))
    if dimensions['evaluation_regime'] == 'Kaggle+MAGE+Outfox':
        mixed_models_by_base_config[(dimensions['scorer_llm'], dimensions['model_family'], dimensions['training_window'], dimensions['evaluation_window'])] = fitted
document_level_metrics = pd.concat(document_metric_frames, ignore_index=True)
document_test_predictions = pd.concat(document_test_prediction_frames, ignore_index=True)
display(document_level_metrics.sort_values(['evaluation_regime', 'scorer_llm', 'evaluation_window', 'model_family', 'model']))

## Whole-text representation comparison

T08 compares four mean-window routes (train32/64 × eval32/64) with one-vector whole_text routes (train32, train64, train-whole_text). Every method is evaluated on the per-scorer/classifier/regime intersection of document IDs.

In [ ]:
window_method_frames = []
for (regime, scorer, family, train_window, eval_window), group in main_window_rows.groupby(
    ['evaluation_regime', 'scorer_llm', 'model_family', 'training_window', 'evaluation_window'], sort=True
):
    docs = aggregate_window_predictions(group, document_threshold=0.5)
    method = f'window_train{train_window}_eval{eval_window}'
    window_method_frames.append(docs.assign(
        evaluation_regime=regime, scorer_llm=scorer, model_family=family,
        training_window=train_window, evaluation_window=eval_window, method_variant=method,
        training_representation=f'window{train_window}', evaluation_representation=f'mean_window{eval_window}',
        aggregation='mean_window_score',
    ))
whole_text_method_frames = []
for regime, scorer, family, train_representation in tqdm(product(
    EVALUATION_REGIMES, SCORERS, MODEL_FAMILIES, WHOLE_TEXT_TRAINING_REPRESENTATIONS,
), desc='Loading whole-text predictions'):
    frame = load_evaluation_window_rows(
        PREDICTION_ROOT, run_id=RUN_ID, scorer_llm=scorer, model_family=family,
        feature_set='simple_similarity', training_regime=TRAINING_REGIME, training_window=train_representation,
        evaluation_regime=regime, evaluation_window='whole_text', partition='test',
    )
    if frame.duplicated(['source_dataset', 'split', 'text_id']).any():
        raise ValueError(f'whole_text is not one-row-per-document: {regime}/{scorer}/{family}/{train_representation}')
    docs = frame.rename(columns={'mg_score': 'document_score'}).copy()
    if not (docs['prediction'].astype(int).eq((docs['document_score'] >= 0.5).astype(int))).all():
        raise ValueError('whole_text artifact predictions do not match fixed score threshold 0.5')
    method = f'whole_text_train{train_representation}'
    whole_text_method_frames.append(docs.assign(
        evaluation_regime=regime, scorer_llm=scorer, model_family=family,
        training_window=train_representation, evaluation_window='whole_text', method_variant=method,
        training_representation=str(train_representation), evaluation_representation='whole_text',
        aggregation='single_whole_text_vector',
    ))
t08_predictions_available = normalize_t08_aggregation_labels(pd.concat([*window_method_frames, *whole_text_method_frames], ignore_index=True, sort=False))
t08_predictions = t08_predictions_available.copy()
t08_predictions, t08_coverage = filter_window_rows_to_common_documents(
    t08_predictions, comparison_group_columns=('evaluation_regime', 'scorer_llm', 'model_family'),
    configuration_columns=('method_variant',),
)
t08_group_columns = ['evaluation_regime', 'scorer_llm', 'model_family', 'method_variant', 'training_representation', 'evaluation_representation', 'aggregation']
t08_metric_rows = []
for keys, group in t08_predictions.groupby(t08_group_columns, dropna=False, sort=True):
    t08_metric_rows.append({**evaluate_document_predictions(group, document_threshold=0.5), **dict(zip(t08_group_columns, keys))})
t08_document_comparison = pd.DataFrame(t08_metric_rows)

word_counts = load_document_word_counts(DATA_ROOT, t08_predictions[['source_dataset', 'split', 'text_id']].drop_duplicates())
t08_with_word_count = t08_predictions.assign(text_id=t08_predictions['text_id'].astype(str)).merge(word_counts, on=['source_dataset', 'split', 'text_id'], how='left', validate='many_to_one')
if t08_with_word_count['word_count'].isna().any():
    raise ValueError('raw word counts are missing for T08 documents')
t08_with_word_count['word_length_bin'] = pd.cut(
    t08_with_word_count['word_count'], bins=[-1, 250, 500, 1000, float('inf')],
    labels=['<=250', '251-500', '501-1000', '>1000'],
)
length_group_columns = [*t08_group_columns, 'word_length_bin']
length_metric_rows = []
for keys, group in t08_with_word_count.groupby(length_group_columns, observed=True, dropna=False, sort=True):
    hg_support = int(group.label.eq(0).sum())
    mg_support = int(group.label.eq(1).sum())
    length_metric_rows.append(dict(zip(length_group_columns, keys), document_count=len(group), hg_support=hg_support, mg_support=mg_support,
        macro_f1=f1_score(group.label, group.prediction, labels=[0,1], average='macro', zero_division=0) if min(hg_support, mg_support) >= 20 else float('nan'),
        status='available' if min(hg_support, mg_support) >= 20 else 'insufficient_class_support_below_20'))
t08_length_metrics = pd.DataFrame(length_metric_rows)
display(t08_document_comparison.sort_values(['evaluation_regime', 'scorer_llm', 'model_family', 'method_variant']))
display(t08_length_metrics.head(25))

## Whole-text representation comparison

T08 compares the four mean-window routes (train32/64 × eval32/64) with one-vector whole_text routes (train32, train64, or train-whole_text). Every method is evaluated on the per-scorer/classifier/regime intersection of document IDs.

In [ ]:
window_method_frames = []
for (regime, scorer, family, train_window, eval_window), group in main_window_rows.groupby(
    ['evaluation_regime', 'scorer_llm', 'model_family', 'training_window', 'evaluation_window'], sort=True
):
    docs = aggregate_window_predictions(group, document_threshold=0.5)
    method = f'window_train{train_window}_eval{eval_window}'
    window_method_frames.append(docs.assign(
        evaluation_regime=regime, scorer_llm=scorer, model_family=family,
        training_window=train_window, evaluation_window=eval_window,
        method_variant=method, training_representation=f'window{train_window}',
        evaluation_representation=f'mean_window{eval_window}', aggregation='mean_window_score',
    ))
whole_text_method_frames = []
for regime, scorer, family, train_representation in product(
    EVALUATION_REGIMES, SCORERS, MODEL_FAMILIES, WHOLE_TEXT_TRAINING_REPRESENTATIONS,
):
    frame = load_evaluation_window_rows(
        PREDICTION_ROOT, run_id=RUN_ID, scorer_llm=scorer, model_family=family,
        feature_set='simple_similarity', training_regime=TRAINING_REGIME, training_window=train_representation,
        evaluation_regime=regime, evaluation_window='whole_text', partition='test',
    )
    if frame.duplicated(['source_dataset', 'split', 'text_id']).any():
        raise ValueError(f'whole_text predictions are not one-row-per-document: {regime}/{scorer}/{family}/{train_representation}')
    docs = frame.rename(columns={'mg_score': 'document_score'}).copy()
    docs['prediction_from_vector'] = (docs['document_score'] >= 0.5).astype(int)
    if not docs['prediction_from_vector'].eq(docs['prediction']).all():
        raise ValueError('whole_text artifact prediction disagrees with score threshold 0.5')
    method = f'whole_text_train{train_representation}'
    whole_text_method_frames.append(docs.assign(
        evaluation_regime=regime, scorer_llm=scorer, model_family=family,
        training_window=train_representation, evaluation_window='whole_text',
        method_variant=method, training_representation=str(train_representation),
        evaluation_representation='whole_text', aggregation='single_whole_text_vector',
    ))
t08_predictions = normalize_t08_aggregation_labels(pd.concat([*window_method_frames, *whole_text_method_frames], ignore_index=True, sort=False))
t08_predictions, t08_coverage = filter_window_rows_to_common_documents(
    t08_predictions, comparison_group_columns=('evaluation_regime', 'scorer_llm', 'model_family'),
    configuration_columns=('method_variant',),
)
t08_metric_rows = []
t08_group_columns = ['evaluation_regime', 'scorer_llm', 'model_family', 'method_variant', 'training_representation', 'evaluation_representation', 'aggregation']
for keys, group in t08_predictions.groupby(t08_group_columns, dropna=False, sort=True):
    metrics = evaluate_document_predictions(group, document_threshold=0.5)
    t08_metric_rows.append({**metrics, **dict(zip(t08_group_columns, keys))})
t08_document_comparison = pd.DataFrame(t08_metric_rows)

word_counts = load_document_word_counts(
    DATA_ROOT, t08_predictions[['source_dataset', 'split', 'text_id']].drop_duplicates(),
)
t08_with_word_count = t08_predictions.assign(text_id=t08_predictions['text_id'].astype(str)).merge(
    word_counts, on=['source_dataset', 'split', 'text_id'], how='left', validate='many_to_one',
)
if t08_with_word_count['word_count'].isna().any():
    raise ValueError('raw word counts are missing for T08 documents')
t08_with_word_count['word_length_bin'] = pd.cut(
    t08_with_word_count['word_count'], bins=[-1, 250, 500, 1000, float('inf')],
    labels=['<=250', '251-500', '501-1000', '>1000'],
)
length_rows = []
length_groups = ['evaluation_regime', 'scorer_llm', 'model_family', 'method_variant', 'word_length_bin']
for keys, group in t08_with_word_count.groupby(length_groups, observed=True, sort=True):
    hg_support = int(group.label.eq(0).sum())
    mg_support = int(group.label.eq(1).sum())
    length_rows.append(dict(zip(length_groups, keys), document_count=len(group), hg_support=hg_support, mg_support=mg_support,
        macro_f1=f1_score(group.label, group.prediction, labels=[0,1], average='macro', zero_division=0) if min(hg_support, mg_support) >= 20 else float('nan'),
        status='available' if min(hg_support, mg_support) >= 20 else 'insufficient_class_support_below_20'))
t08_length_metrics = pd.DataFrame(length_rows)
display(t08_document_comparison.sort_values(['evaluation_regime', 'scorer_llm', 'model_family', 'method_variant']))
display(t08_length_metrics.head(30))

## Document-level BBC-only and Name That LLM results

Apply the Mixed-validation document models to target-domain documents without refitting. BBC-only is HG-only and reports FPR. Name That LLM is MG-only and reports detection rate by source LLM after the 250-token filter.

In [ ]:
bbc_document_prediction_frames = []
for scorer, family, train_window, eval_window in tqdm(product(SCORERS, MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS), desc='Scoring BBC windows'):
    base_windows = bbc_windows.loc[
        bbc_windows['scorer_llm'].eq(scorer) & bbc_windows['model_family'].eq(family)
        & bbc_windows['training_window'].eq(train_window) & bbc_windows['evaluation_window'].eq(eval_window)
    ]
    profiles = build_document_feature_table(base_windows)
    fitted = mixed_models_by_base_config[(scorer, family, train_window, eval_window)]
    predictions = score_document_models(profiles, fitted)
    bbc_document_prediction_frames.append(predictions.assign(scorer_llm=scorer, model_family=family, training_window=train_window, evaluation_window=eval_window))
bbc_document_predictions = pd.concat(bbc_document_prediction_frames, ignore_index=True)
bbc_document_fprs = single_class_rates(
    bbc_document_predictions, group_columns=('scorer_llm', 'model_family', 'training_window', 'evaluation_window', 'model'), unit_name='document',
)

available_n2llm_scorers = tuple(sorted(n2llm_windows['scorer_llm'].dropna().unique()))
n2llm_document_prediction_frames = []
for scorer, family, train_window, eval_window in tqdm(product(available_n2llm_scorers, MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS), desc='Scoring Name That LLM documents'):
    base_windows = n2llm_windows.loc[
        n2llm_windows['scorer_llm'].eq(scorer) & n2llm_windows['model_family'].eq(family)
        & n2llm_windows['training_window'].eq(train_window)
        & n2llm_windows['evaluation_window'].eq(eval_window)
    ]
    profiles = build_document_feature_table(base_windows)
    profile_metadata = base_windows[['text_id', 'id', 'source_llm', 'token_count']].drop_duplicates('text_id')
    profiles['text_id'] = profiles['text_id'].astype(str)
    profiles = profiles.merge(profile_metadata.assign(text_id=lambda frame: frame['text_id'].astype(str)), on='text_id', validate='one_to_one')
    fitted = mixed_models_by_base_config[(scorer, family, train_window, eval_window)]
    predictions = score_document_models(profiles, fitted)
    n2llm_document_prediction_frames.append(predictions.assign(scorer_llm=scorer, model_family=family, training_window=train_window, evaluation_window=eval_window))
n2llm_document_predictions = pd.concat(n2llm_document_prediction_frames, ignore_index=True)
n2llm_document_detection = single_class_rates(
    n2llm_document_predictions, group_columns=('scorer_llm', 'model_family', 'training_window', 'evaluation_window', 'model'), unit_name='document',
)
n2llm_document_detection_by_source = summarize_mg_detection(
    n2llm_document_predictions, group_columns=('source_llm', 'scorer_llm', 'model_family', 'training_window', 'evaluation_window'),
)
display(bbc_document_fprs)
display(n2llm_document_detection)
display(n2llm_document_detection_by_source)

In [ ]:
window_level_metrics = pd.concat([
    window_level_main_metrics.assign(block='W', dataset='main_test'),
    feature_window_metrics.assign(block='F', dataset='main_test'),
    training_regime_metrics.assign(block='S', dataset='main_test', evaluation_regime='Kaggle+MAGE+Outfox'),
    semeval_window_metrics.assign(block='SemEval_window_binary', dataset='SemEval_train_pure_windows', evaluation_regime='SemEval'),
], ignore_index=True, sort=False)
document_level_metrics = document_level_metrics.assign(
    metric_scope='held_out_binary_test', false_positive_rate=lambda frame: frame.fp / (frame.tn + frame.fp),
)

single_class_rates_table = pd.concat([
    bbc_window_rates.assign(dataset='BBC-only', evaluation_unit='window'),
    n2llm_window_rates.assign(dataset='Name That LLM >=250 tokens', evaluation_unit='window'),
    bbc_document_fprs.assign(dataset='BBC-only', evaluation_unit='document'),
    bbc_whole_text_rates.assign(dataset='BBC-only', evaluation_unit='document', evaluation_representation='whole_text'),
    n2llm_document_detection.assign(dataset='Name That LLM >=250 tokens', evaluation_unit='document'),
], ignore_index=True, sort=False)

# T01–T10 report views. Full rows retain all configuration dimensions.
table_t04_training_composition = training_regime_metrics.sort_values(['training_regime', 'model_family', 'evaluation_window'])
table_t05_feature_sets = feature_window_metrics.sort_values(['scorer_llm', 'training_window', 'feature_set', 'model_family'])
table_t06_window_transfer = window_level_main_metrics.pivot(
    index=['scorer_llm', 'model_family', 'training_window', 'evaluation_window'],
    columns='evaluation_regime', values='macro_f1',
).reset_index()
table_t06_window_transfer.columns.name = None
table_t06_window_transfer = table_t06_window_transfer.rename(columns={
    'Kaggle+MAGE+Outfox': 'f1_mixed', 'Kaggle+MAGE+Outfox+BBC': 'f1_mixed_bbc',
})
table_t06_window_transfer['delta_f1_mixed_bbc_minus_mixed'] = table_t06_window_transfer.f1_mixed_bbc - table_t06_window_transfer.f1_mixed
table_t06_window_transfer['cohort_comparison'] = 'regimes have separately formed samples; descriptive delta only'
table_t07_document_mean = document_level_metrics.loc[document_level_metrics.model.eq('mean_score_baseline')].sort_values(
    ['evaluation_regime', 'scorer_llm', 'model_family', 'training_window', 'evaluation_window']
)
table_t08_whole_text = t08_document_comparison.sort_values(['evaluation_regime', 'scorer_llm', 'model_family', 'method_variant'])
table_t09_external = single_class_rates_table.sort_values(['dataset', 'evaluation_unit', 'scorer_llm', 'model_family', 'training_window', 'evaluation_window', 'model'])

semeval_run_root = DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / RUN_ID
semeval_region_summary = pd.read_csv(semeval_run_root / 'semeval_summary.csv')
from notebooks.utils.semeval_boundary_utils import summarize_boundary_predictions
semeval_boundary_predictions = pd.read_csv(semeval_run_root / 'semeval_boundary_predictions.csv')
semeval_boundary_summary_existing = summarize_boundary_predictions(
    semeval_boundary_predictions,
    group_columns=(
        'scorer_llm', 'feature_set', 'training_regime', 'training_window', 'model_family',
        'window_length', 'evaluation_dataset', 'semeval_split', 'model',
    ),
)
semeval_filter = (
    semeval_region_summary.feature_set.eq('simple_similarity')
    & semeval_region_summary.training_regime.eq(TRAINING_REGIME)
    & semeval_region_summary.evaluation_dataset.eq('semeval2024_task8_subtask_c_train')
    & semeval_region_summary.semeval_split.eq('train')
    & semeval_region_summary.scorer_llm.isin(SCORERS)
    & semeval_region_summary.model_family.isin(MODEL_FAMILIES)
    & semeval_region_summary.training_window.astype(str).isin(['32', '64'])
    & semeval_region_summary.window_length.astype(str).eq(semeval_region_summary.training_window.astype(str))
)
semeval_boundary_filter = (
    semeval_boundary_summary_existing.feature_set.eq('simple_similarity')
    & semeval_boundary_summary_existing.training_regime.eq(TRAINING_REGIME)
    & semeval_boundary_summary_existing.evaluation_dataset.eq('semeval2024_task8_subtask_c_train')
    & semeval_boundary_summary_existing.semeval_split.eq('train')
    & semeval_boundary_summary_existing.scorer_llm.isin(SCORERS)
    & semeval_boundary_summary_existing.model_family.isin(MODEL_FAMILIES)
    & semeval_boundary_summary_existing.training_window.astype(str).isin(['32', '64'])
    & semeval_boundary_summary_existing.window_length.astype(str).eq(semeval_boundary_summary_existing.training_window.astype(str))
)
t10_keys = ['scorer_llm', 'feature_set', 'training_regime', 'training_window', 'model_family', 'window_length', 'evaluation_dataset', 'semeval_split']
table_t10_localization = semeval_region_summary.loc[semeval_filter].merge(
    semeval_boundary_summary_existing.loc[semeval_boundary_filter], on=t10_keys, how='outer', validate='one_to_one', suffixes=('_region', '_boundary'),
)

# T03: evaluated sample sizes by source/class and model configuration.
t03_main = main_window_rows.groupby(
    ['evaluation_regime', 'scorer_llm', 'model_family', 'training_window', 'evaluation_window', 'source_dataset', 'split', 'label'], as_index=False
).agg(document_count=('text_id', 'nunique'), window_count=('text_id', 'size')).assign(dataset_role='main_test', length_rule='source run eligibility')
t03_bbc = bbc_windows.groupby(
    ['scorer_llm', 'model_family', 'training_window', 'evaluation_window', 'source_dataset', 'split', 'label'], as_index=False
).agg(document_count=('text_id', 'nunique'), window_count=('text_id', 'size')).assign(dataset_role='external_BBC_HG', length_rule='available prepared partition')
t03_semeval = semeval_windows.groupby(
    ['scorer_llm', 'model_family', 'training_window', 'evaluation_window', 'source_dataset', 'split', 'gold_label'], as_index=False
).agg(document_count=('text_id', 'nunique'), window_count=('text_id', 'size')).rename(columns={'gold_label': 'label'}).assign(dataset_role='SemEval_train_pure_windows', length_rule='gold-boundary pure windows')
t03_n2llm = n2llm_windows.groupby(
    ['scorer_llm', 'model_family', 'training_window', 'evaluation_window', 'source_dataset', 'split', 'source_llm', 'label'], as_index=False
).agg(document_count=('text_id', 'nunique'), window_count=('text_id', 'size')).assign(dataset_role='Name That LLM MG train cohort', length_rule='Gemma token_count >= 250; common 1800-ID cohort')
t03_whole = t08_predictions.loc[t08_predictions.evaluation_representation.eq('whole_text')].groupby(
    ['evaluation_regime', 'scorer_llm', 'model_family', 'training_representation', 'source_dataset', 'split', 'label'], as_index=False
).agg(document_count=('text_id', 'nunique')).assign(window_count=1, dataset_role='whole_text_test', length_rule='whole_text representation')
table_t03_dataset_counts = pd.concat([t03_main, t03_bbc, t03_semeval, t03_n2llm, t03_whole], ignore_index=True, sort=False)

# Configuration coverage and explicit missing computations.
expected_w = pd.DataFrame([dict(zip(['scorer_llm','model_family','training_window','evaluation_window','evaluation_regime'], x)) for x in product(SCORERS, MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS, EVALUATION_REGIMES)])
coverage_w = build_configuration_coverage(expected_w, window_level_main_metrics, block='W', key_columns=('scorer_llm','model_family','training_window','evaluation_window','evaluation_regime'), missing_reason='test window prediction artifacts missing')
expected_f = pd.DataFrame([{'scorer_llm': scorer, 'model_family': family, 'feature_set': feature_set, 'training_window': window, 'evaluation_window': window} for scorer, family, feature_set, window in product(SCORERS, MODEL_FAMILIES, FEATURE_SETS, WINDOWS)])
coverage_f = build_configuration_coverage(expected_f, feature_window_metrics, block='F', key_columns=('scorer_llm','model_family','feature_set','training_window','evaluation_window'), missing_reason='feature-set test predictions missing')
expected_s = pd.DataFrame([{'scorer_llm': scorer, 'model_family': family, 'training_regime': regime, 'training_window': window, 'evaluation_window': window} for scorer, family, regime, window in product(['gemma_3_270m_it'], MODEL_FAMILIES, TRAINING_REGIMES, WINDOWS)])
coverage_s = build_configuration_coverage(expected_s, training_regime_metrics, block='S', key_columns=('scorer_llm','model_family','training_regime','training_window','evaluation_window'), missing_reason='training-composition test predictions missing')
expected_d = expected_w.copy()
coverage_d = build_configuration_coverage(expected_d, document_level_metrics, block='D', key_columns=('scorer_llm','model_family','training_window','evaluation_window','evaluation_regime'), missing_reason='validation/test document profiles missing')
expected_h = pd.DataFrame([dict(zip(['scorer_llm','model_family','training_representation','evaluation_regime'], x)) for x in product(SCORERS, MODEL_FAMILIES, WHOLE_TEXT_TRAINING_REPRESENTATIONS, EVALUATION_REGIMES)])
available_h = t08_document_comparison.loc[t08_document_comparison.aggregation.eq('single_whole_text_vector'), ['scorer_llm','model_family','training_representation','evaluation_regime']].drop_duplicates()
coverage_h = build_configuration_coverage(expected_h, available_h, block='H', key_columns=('scorer_llm','model_family','training_representation','evaluation_regime'), missing_reason='whole_text evaluation artifact missing')
expected_l = pd.DataFrame([{'scorer_llm': scorer, 'model_family': family, 'training_window': window, 'evaluation_window': window} for scorer, family, window in product(SCORERS, MODEL_FAMILIES, WINDOWS)])
available_l = table_t10_localization[['scorer_llm','model_family','training_window','window_length']].rename(columns={'window_length':'evaluation_window'}).drop_duplicates()
coverage_l = build_configuration_coverage(expected_l, available_l, block='L', key_columns=('scorer_llm','model_family','training_window','evaluation_window'), missing_reason='SemEval localization summary missing')
expected_ebbc = pd.DataFrame([dict(zip(['scorer_llm','model_family','training_window','evaluation_window'], x)) for x in product(SCORERS, MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS)])
available_ebbc = bbc_window_rates[['scorer_llm','model_family','training_window','evaluation_window']].drop_duplicates()
coverage_ebbc = build_configuration_coverage(expected_ebbc, available_ebbc, block='E_BBC_windows', key_columns=('scorer_llm','model_family','training_window','evaluation_window'), missing_reason='BBC prepared HG windows or stored model missing')
expected_en2 = pd.DataFrame([dict(zip(['scorer_llm','model_family','training_window','evaluation_window'], x)) for x in product(['gemma_3_270m_it','littlelamb'], MODEL_FAMILIES, TRAIN_WINDOWS, EVALUATION_WINDOWS)])
available_en2 = n2llm_window_rates[['scorer_llm','model_family','training_window','evaluation_window']].drop_duplicates()
coverage_en2 = build_configuration_coverage(expected_en2, available_en2, block='E_Name_That_LLM_windows', key_columns=('scorer_llm','model_family','training_window','evaluation_window'), missing_reason='Name That LLM prepared windows unavailable for scorer')
expected_ebbc_whole = pd.DataFrame([dict(zip(['scorer_llm','model_family','training_representation'], x)) for x in product(SCORERS, MODEL_FAMILIES, WHOLE_TEXT_TRAINING_REPRESENTATIONS)])
available_ebbc_whole = bbc_whole_text_rates[['scorer_llm','model_family','training_window']].rename(columns={'training_window':'training_representation'}).drop_duplicates()
coverage_ebbc_whole = build_configuration_coverage(expected_ebbc_whole, available_ebbc_whole, block='E_BBC_whole_text', key_columns=('scorer_llm','model_family','training_representation'), missing_reason='BBC whole_text prepared partition or model missing')
expected_en2_whole = pd.DataFrame([dict(zip(['scorer_llm','model_family','training_representation'], x)) for x in product(['gemma_3_270m_it','littlelamb'], MODEL_FAMILIES, WHOLE_TEXT_TRAINING_REPRESENTATIONS)])
coverage_en2_whole = build_configuration_coverage(expected_en2_whole, pd.DataFrame(columns=expected_en2_whole.columns), block='E_Name_That_LLM_whole_text', key_columns=('scorer_llm','model_family','training_representation'), missing_reason='No prepared Name That LLM whole_text partitions in current data')
configuration_coverage = pd.concat([coverage_w,coverage_f,coverage_s,coverage_d,coverage_h,coverage_l,coverage_ebbc,coverage_en2,coverage_ebbc_whole,coverage_en2_whole], ignore_index=True, sort=False)
missing_computations = configuration_coverage.loc[configuration_coverage.status.eq('unavailable')].copy()

# Full provenance cohort manifest: include both retained and excluded document keys.
cohort_parts = [
    build_cohort_manifest(main_window_rows_available,main_window_rows,configuration_columns=('scorer_llm','evaluation_regime','model_family','training_window','evaluation_window'),exclusion_reason='not_in_common_test_cohort'),
    build_cohort_manifest(feature_window_rows_available,feature_window_rows,configuration_columns=('scorer_llm','evaluation_regime','model_family','feature_set','training_window','evaluation_window'),exclusion_reason='not_in_common_feature_set_cohort'),
    build_cohort_manifest(training_regime_rows_available,training_regime_rows,configuration_columns=('scorer_llm','evaluation_regime','model_family','training_regime','training_window','evaluation_window'),exclusion_reason='not_in_common_training_regime_cohort'),
    build_cohort_manifest(pd.concat(list(test_by_config.values()),ignore_index=True),pd.concat(list(test_common.values()),ignore_index=True),configuration_columns=('scorer_llm','evaluation_regime','model_family','training_window','evaluation_window'),exclusion_reason='not_in_common_document_test_cohort'),
    build_cohort_manifest(t08_predictions_available,t08_predictions,configuration_columns=('scorer_llm','evaluation_regime','model_family','method_variant'),exclusion_reason='not_in_common_seven_method_cohort'),
    build_cohort_manifest(bbc_window_rows_available,bbc_windows,configuration_columns=('scorer_llm','model_family','training_window','evaluation_window'),exclusion_reason='not_in_common_BBC_classifier_window_cohort'),
    build_cohort_manifest(semeval_scored_frames[0] if len(semeval_scored_frames)==1 else pd.concat(semeval_scored_frames,ignore_index=True),semeval_windows,configuration_columns=('scorer_llm','model_family','training_window','evaluation_window'),exclusion_reason='not_in_common_pure_SemEval_window_cohort'),
    build_cohort_manifest(n2llm_window_rows_available,n2llm_windows,configuration_columns=('scorer_llm','model_family','training_window','evaluation_window'),exclusion_reason='outside_selected_or_below_minimum_length_cohort'),
]
cohorts_manifest = pd.concat(cohort_parts, ignore_index=True, sort=False)

# Report-view and full tables.
table_t04_training_composition = training_regime_metrics.sort_values(['training_regime','model_family','evaluation_window'])
table_t05_feature_sets = feature_window_metrics.sort_values(['scorer_llm','training_window','feature_set','model_family'])
table_t06_window_transfer = window_level_main_metrics.pivot(index=['scorer_llm','model_family','training_window','evaluation_window'],columns='evaluation_regime',values='macro_f1').reset_index()
table_t06_window_transfer.columns.name = None
table_t06_window_transfer = table_t06_window_transfer.rename(columns={'Kaggle+MAGE+Outfox':'f1_mixed','Kaggle+MAGE+Outfox+BBC':'f1_mixed_bbc'})
table_t06_window_transfer['delta_f1_mixed_bbc_minus_mixed'] = table_t06_window_transfer.f1_mixed_bbc - table_t06_window_transfer.f1_mixed
table_t06_window_transfer['cohort_relationship'] = 'separately formed regime cohorts; descriptive delta only'
table_t07_document_mean = document_level_metrics.loc[document_level_metrics.model.eq('mean_score_baseline')].sort_values(['evaluation_regime','scorer_llm','model_family','training_window','evaluation_window'])
table_t08_whole_text = t08_document_comparison.sort_values(['evaluation_regime','scorer_llm','model_family','method_variant'])
table_t09_external = single_class_rates_table.sort_values(['dataset','evaluation_unit','scorer_llm','model_family','training_window','evaluation_window','model'])

table_t01_feature_sets.to_csv(TABLE_ROOT/'T01_feature_sets.csv',index=False)
table_t02_evaluation_units.to_csv(TABLE_ROOT/'T02_evaluation_units.csv',index=False)
table_t03_dataset_counts.to_csv(TABLE_ROOT/'T03_dataset_counts.csv',index=False)
table_t04_training_composition.to_csv(TABLE_ROOT/'T04_training_composition.csv',index=False)
table_t05_feature_sets.to_csv(TABLE_ROOT/'T05_feature_sets.csv',index=False)
window_level_metrics.to_csv(TABLE_ROOT/'T06_window_metrics_long.csv',index=False)
table_t06_window_transfer.to_csv(TABLE_ROOT/'T06_window_transfer_report_view.csv',index=False)
table_t07_document_mean.to_csv(TABLE_ROOT/'T07_document_mean_window.csv',index=False)
table_t08_whole_text.to_csv(TABLE_ROOT/'T08_window_vs_whole_text.csv',index=False)
table_t09_external.to_csv(TABLE_ROOT/'T09_external_document_results.csv',index=False)
table_t10_localization.to_csv(TABLE_ROOT/'T10_semeval_localization.csv',index=False)
t08_length_metrics.to_csv(TABLE_ROOT/'T08_document_length_metrics.csv',index=False)
document_level_metrics.to_csv(APPENDIX_ROOT/'document_level_all_methods_long.csv',index=False)
window_level_metrics.to_csv(APPENDIX_ROOT/'window_level_all_configs_long.csv',index=False)
single_class_rates_table.to_csv(APPENDIX_ROOT/'single_class_rates_all_configs.csv',index=False)
configuration_coverage.to_csv(MANIFEST_ROOT/'configuration_coverage.csv',index=False)
missing_computations.to_csv(MANIFEST_ROOT/'missing_computations.csv',index=False)
manifest_object_columns = cohorts_manifest.select_dtypes(include='object').columns
cohorts_manifest[manifest_object_columns] = cohorts_manifest[manifest_object_columns].astype('string')
cohorts_manifest.to_parquet(MANIFEST_ROOT/'cohorts.parquet',index=False)
semeval_exclusion_summary.to_csv(MANIFEST_ROOT/'semeval_boundary_exclusions.csv',index=False)

run_manifest = {
    'source_run_id': RUN_ID, 'report_run_id': REPORT_RUN_ID, 'feature_sets': FEATURE_SETS,
    'scorers': SCORERS, 'model_families': MODEL_FAMILIES, 'training_regimes': TRAINING_REGIMES,
    'training_windows': TRAIN_WINDOWS, 'evaluation_windows': EVALUATION_WINDOWS,
    'sample_mode': 'common document IDs within comparison group when available',
    'document_threshold': 0.5, 'window_threshold': 0.5,
    'semeval_boundary_rule': 'pure windows only; boundary-straddling and token-alignment failures counted/excluded',
    'name_that_llm_cohort_limit': N2LLM_LIMIT, 'name_that_llm_min_token_count': MIN_TOKEN_COUNT,
    'validation_model_selection_caveat': True, 'first_stage_training_or_llm_inference_run': False,
    'matrix_counts': {'W':64,'F':64,'S':20,'D':64,'H':48,'L':16},
    'available_configuration_count': int(configuration_coverage.status.eq('available').sum()),
    'unavailable_configuration_count': int(configuration_coverage.status.eq('unavailable').sum()),
}
(MANIFEST_ROOT/'run_manifest.json').write_text(json.dumps(run_manifest,indent=2,default=str)+'\n',encoding='utf-8')

README_TEXT = f'''# Chapter 3 comparison report\n\nSource run: `{RUN_ID}`. This report uses cached models/features/predictions only and performs no LLM inference or first-stage training.\n\nTables T01-T10 and figures F01-F08 follow the Chapter 3 specification. `configuration_coverage.csv` and `missing_computations.csv` are authoritative for unavailable cells. Mixed+BBC is a separately formed cohort; its delta from Mixed is descriptive, not a paired causal effect unless cohort manifests establish a shared ID set.\n\nBBC-only contains HG only and is reported with FPR. Name That LLM is MG-only and is reported with MG detection rate on the selected 1,800-document cohort and token_count >= {MIN_TOKEN_COUNT}. SemEval uses the available train split; its window macro-F1 excludes boundary-crossing windows, and its localization metrics are not official test-set metrics. Document meta-models fit validation profiles and evaluate test once; validation was used in first-stage model selection and no OOF predictions are available.\n\nMacro-F1 means are not averaged across unrelated datasets. Full configuration-level rows and cohort IDs are in the appendix/manifests. Figures use 300 dpi PNG plus SVG.\n'''
(OUTPUT_ROOT/'README.md').write_text(README_TEXT,encoding='utf-8')
print(f'Chapter 3 report written to {OUTPUT_ROOT}; expected grids W/F/S/D/H/L and external coverage are listed in manifests.')
display(configuration_coverage.groupby(['block','status']).size().rename('configuration_count').reset_index())
display(missing_computations)